In [1]:
import openmc
from inputs import *
from aux_functions import *
import numpy as np
import matplotlib.pyplot as plt
import os

/home/jpontes/miniforge/envs/openmc-env/lib/python3.13/site-packages/openmc/material.py:825: UserWarning: A uranium enrichment of 19.75 was given for Material ID="1". OpenMC assumes the U234/U235 mass ratio is constant at 0.008, which is only valid at low enrichments. Consider setting the isotopic composition manually for enrichments over 5%.
  warnings.warn(msg)


#### TRISO homogenization

In [2]:
#Hole particle volume

In [3]:
TRISO_materials_list = [UO2, Buffer, PyC, SiC, PyC]
triso_universe=TRISOp(TRISO_radii,TRISO_materials_list)

geometry = openmc.Geometry(triso_universe)

vol_calc = openmc.VolumeCalculation([triso_universe],1000000,lower_left=triso_universe.bounding_box[0],upper_right=triso_universe.bounding_box[1])

settings = openmc.Settings()
settings.volume_calculations = [vol_calc]
settings.temperature = {'default': T_avg, 'method': 'interpolation'}

materials = openmc.Materials(TRISO_materials_list)
materials.cross_sections = xs_path

materials.export_to_xml()
settings.export_to_xml()
geometry.export_to_xml()

In [4]:
openmc.calculate_volumes()
os.rename('volume_1.h5','volume_triso_particle.h5')

                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

In [5]:
vol_calc.load_results('volume_triso_particle.h5')
triso_volume = vol_calc.volumes[1].nominal_value

In [6]:
vol_calc.volumes

{1: 0.00032709692790787524+/-3.1217271962348987e-07}

In [7]:
data_frame = vol_calc.atoms_dataframe
nuclide_atoms_dict = dict(zip(data_frame['Nuclide'], data_frame['Atoms']))
atoms_list=[]
for nuclide in nuclide_atoms_dict:
    atoms = nuclide_atoms_dict[nuclide]
    atoms_list.append(atoms)

In [8]:
triso_universe.volume=vol_calc.volumes[1].nominal_value

In [9]:
os.rename('settings.xml','settings_full_triso.xml')

In [10]:
#Each layer volume

In [11]:
cells_dict = triso_universe.cells
triso_cells = []
for cell in cells_dict:
    triso_cells.append(cells_dict[cell])

In [12]:
vol_calc2 = openmc.VolumeCalculation(triso_cells,1000000,lower_left=triso_universe.bounding_box[0],upper_right=triso_universe.bounding_box[1])

settings = openmc.Settings()
settings.volume_calculations = [vol_calc2]
settings.temperature = {'default': T_avg, 'method': 'interpolation'}

settings.export_to_xml()

In [13]:
openmc.calculate_volumes()
os.rename('volume_1.h5','volume_triso_layers.h5')

                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

In [14]:
vol_calc2.load_results('volume_triso_layers.h5')
layers_volumes = vol_calc2.volumes
volume_fractions = []
for layer in layers_volumes:
    volume_fraction = (layers_volumes[layer].nominal_value)/triso_volume
    volume_fractions.append(volume_fraction)

In [15]:
vols=[]
for i in range(1,6):
    vol = vol_calc2.volumes[i].nominal_value
    vols.append(vol)
sum(vols)

0.00032709692790787524

In [16]:
volume_fraction_dict = {}
for i in range(len(materials[0:5])):
    volume_fraction_dict[materials[i].name] = volume_fractions[i]

## Creating json file

In [17]:
import json

In [18]:
with open('volume_fraction_dict.json', 'w') as f:
    json.dump(volume_fractions, f)